# 多日期久期验证分析

本notebook对多个日期的久期计算结果进行验证分析，按不同场景（全量/剔除边界）统计误差指标。

In [5]:
from WindPy import w
import pandas as pd
import numpy as np
import warnings
from tqdm import tqdm

warnings.filterwarnings('ignore')

# 启动Wind
w.start()
print("Wind连接成功")

Wind连接成功


In [6]:
# 分析日期列表
DATES = ['20231231', '20240630', '20250630', '20251231']

print(f"将分析以下日期: {DATES}")

将分析以下日期: ['20231231', '20240630', '20250630', '20251231']


In [7]:
def get_wind_duration_batch(fund_codes, rpt_date, batch_size=100):
    """
    批量获取Wind披露的基金久期
    
    参数:
        fund_codes: 基金代码列表
        rpt_date: 报告日期 'YYYYMMDD'
        batch_size: 每批获取的数量
    
    返回:
        dict: {fund_code: wind_duration}
    """
    wind_durations = {}
    total = len(fund_codes)
    
    for i in range(0, total, batch_size):
        batch = fund_codes[i:i+batch_size]
        
        codes_str = ','.join(batch)
        try:
            data = w.wss(codes_str, "risk_durationupdate", f"rptDate={rpt_date}")
            if data.ErrorCode == 0:
                for j, code in enumerate(batch):
                    if j < len(data.Data[0]):
                        value = data.Data[0][j]
                        if value is not None and not (isinstance(value, float) and np.isnan(value)):
                            wind_durations[code] = float(value)
                        else:
                            wind_durations[code] = None
                    else:
                        wind_durations[code] = None
        except Exception as e:
            for code in batch:
                wind_durations[code] = None
    
    return wind_durations

def calc_type_stats(df, fund_type, bond_type):
    """
    计算特定类型的统计指标
    
    返回dict包含: 样本数, 平均绝对误差, 误差标准差, 平均计算久期, 平均Wind久期
    """
    mask = (df['fund_type'] == fund_type) & (df['bond_type'] == bond_type)
    subset = df[mask]
    
    if len(subset) == 0:
        return None
    
    fund_type_name = '短期' if fund_type == 'short' else '中长期'
    bond_type_name = '信用债' if bond_type == 'credit' else '利率债'
    
    return {
        '类型': f"{fund_type_name}{bond_type_name}",
        'fund_type': fund_type,
        'bond_type': bond_type,
        '样本数': len(subset),
        '平均绝对误差': subset['duration_error'].abs().mean(),
        '误差标准差': subset['duration_error'].std(),
        '平均计算久期': subset['duration'].mean(),
        '平均Wind久期': subset['wind_duration'].mean()
    }

print("辅助函数定义完成")

辅助函数定义完成


In [8]:
# 存储所有结果
all_results = []

# 类型组合定义
TYPE_COMBOS = [
    ('short', 'credit'),
    ('short', 'rate'),
    ('medium_long', 'credit'),
    ('medium_long', 'rate')
]

for date in tqdm(DATES, desc="处理日期"):
    print(f"\n{'='*60}")
    print(f"正在分析日期: {date}")
    print(f"{'='*60}")
    
    # 1. 加载计算结果
    result_file = f'./output/久期详细结果_linear_{date}.xlsx'
    result_df = pd.read_excel(result_file)
    result_df = result_df.rename(columns={'Unnamed: 0': 'fund_code'})
    print(f"加载基金数量: {len(result_df)}")
    
    # 2. 获取Wind真实久期
    fund_codes = result_df['fund_code'].tolist()
    wind_durations = get_wind_duration_batch(fund_codes, rpt_date=date)
    valid_count = sum(1 for v in wind_durations.values() if v is not None)
    print(f"成功获取Wind久期: {valid_count}/{len(fund_codes)}")
    
    result_df['wind_duration'] = result_df['fund_code'].map(wind_durations)
    result_df['duration_error'] = result_df['duration'] - result_df['wind_duration']
    result_df['duration_pct_error'] = (result_df['duration_error'] / result_df['wind_duration']) * 100
    valid_df = result_df[result_df['wind_duration'].notna()].copy()
    print(f"有效数据量: {len(valid_df)}")
    
    # 3. 读取迭代日志，识别边界基金
    log_file = f'./output/久期迭代日志_{date}.xlsx'
    try:
        df_detail = pd.read_excel(log_file, sheet_name='迭代详情')
        df_bnd = df_detail[df_detail['boundary_type'].isin(['upper', 'lower'])].copy()
        
        if len(df_bnd) > 0:
            last_boundary = df_bnd.sort_values('round').groupby('fund_code')['boundary_type'].last()
            upper_funds = set(last_boundary[last_boundary == 'upper'].index.tolist())
            lower_funds = set(last_boundary[last_boundary == 'lower'].index.tolist())
            boundary_funds = upper_funds.union(lower_funds)
            print(f"触发上界: {len(upper_funds)}, 触发下界: {len(lower_funds)}")
        else:
            upper_funds = set()
            lower_funds = set()
            boundary_funds = set()
            print("无边界触发记录")
    except Exception as e:
        print(f"读取迭代日志出错: {e}")
        upper_funds = set()
        lower_funds = set()
        boundary_funds = set()
    
    # 4. 创建4种场景的数据集
    scenarios = {
        '全量': valid_df.copy(),
        '剔除上界': valid_df[~valid_df['fund_code'].isin(upper_funds)].copy(),
        '剔除下界': valid_df[~valid_df['fund_code'].isin(lower_funds)].copy(),
        '剔除上下界': valid_df[~valid_df['fund_code'].isin(boundary_funds)].copy()
    }
    
    # 5. 对每种场景按类型统计
    for scenario_name, scenario_df in scenarios.items():
        print(f"  {scenario_name}: n={len(scenario_df)}")
        
        for fund_type, bond_type in TYPE_COMBOS:
            stats = calc_type_stats(scenario_df, fund_type, bond_type)
            if stats:
                stats['日期'] = date
                stats['场景'] = scenario_name
                all_results.append(stats)

print(f"\n分析完成! 共收集 {len(all_results)} 条统计结果")

处理日期:   0%|          | 0/4 [00:00<?, ?it/s]


正在分析日期: 20231231
加载基金数量: 1274
成功获取Wind久期: 1241/1274
有效数据量: 1241


处理日期:  25%|██▌       | 1/4 [00:10<00:31, 10.51s/it]

触发上界: 317, 触发下界: 299
  全量: n=1241
  剔除上界: n=931
  剔除下界: n=959
  剔除上下界: n=649

正在分析日期: 20240630
加载基金数量: 1371
成功获取Wind久期: 1347/1371
有效数据量: 1347


处理日期:  50%|█████     | 2/4 [00:17<00:16,  8.30s/it]

触发上界: 342, 触发下界: 573
  全量: n=1347
  剔除上界: n=1008
  剔除下界: n=787
  剔除上下界: n=448

正在分析日期: 20250630
加载基金数量: 1507
成功获取Wind久期: 1479/1507
有效数据量: 1479


处理日期:  75%|███████▌  | 3/4 [00:24<00:07,  7.96s/it]

触发上界: 549, 触发下界: 634
  全量: n=1479
  剔除上界: n=937
  剔除下界: n=863
  剔除上下界: n=321

正在分析日期: 20251231
加载基金数量: 1563
成功获取Wind久期: 1552/1563
有效数据量: 1552


处理日期: 100%|██████████| 4/4 [00:32<00:00,  8.15s/it]

触发上界: 700, 触发下界: 594
  全量: n=1552
  剔除上界: n=858
  剔除下界: n=963
  剔除上下界: n=269

分析完成! 共收集 64 条统计结果


In [9]:
# 将所有结果转换为DataFrame
results_df = pd.DataFrame(all_results)

# 选择显示的列
display_cols = ['日期', '场景', '类型', '样本数', '平均绝对误差', '误差标准差', '平均计算久期', '平均Wind久期']
summary_df = results_df[display_cols].copy()

# 确保场景按指定顺序排列
scenario_order = ['全量', '剔除上界', '剔除下界', '剔除上下界']
summary_df['场景'] = pd.Categorical(summary_df['场景'], categories=scenario_order, ordered=True)

# 排序
summary_df = summary_df.sort_values(['日期', '场景', '类型'])

# 显示汇总DataFrame
summary_df

,日期,场景,类型,样本数,平均绝对误差,误差标准差,平均计算久期,平均Wind久期
2,20231231,全量,中长期信用债,740,1.907084,3.782848,3.413033,1.809411
3,20231231,全量,中长期利率债,186,2.078886,3.298391,4.152896,2.608245
0,20231231,全量,短期信用债,288,0.314119,0.457079,0.758944,0.899908
1,20231231,全量,短期利率债,27,0.710522,0.904630,0.932345,1.426726
6,20231231,剔除上界,中长期信用债,555,0.984395,1.679586,2.382151,1.775355
...,...,...,...,...,...,...,...,...
57,20251231,剔除下界,短期利率债,20,0.437867,0.444184,1.970066,1.684604
62,20251231,剔除上下界,中长期信用债,122,1.136209,2.423382,3.085059,2.356010
63,20251231,剔除上下界,中长期利率债,41,2.229384,1.813183,7.819748,5.622434
60,20251231,剔除上下界,短期信用债,103,0.402195,0.404645,1.317544,1.026673


In [10]:
# 保存完整结果到Excel
results_df.to_excel('./output/久期验证多日期汇总结果.xlsx', index=False)
summary_df.to_excel('./output/久期验证多日期汇总展示.xlsx', index=False)

print("结果已保存:")
print("  - ./output/久期验证多日期汇总结果.xlsx (完整数据)")
print("  - ./output/久期验证多日期汇总展示.xlsx (展示用)")

结果已保存:
  - ./output/久期验证多日期汇总结果.xlsx (完整数据)
  - ./output/久期验证多日期汇总展示.xlsx (展示用)
